# Module 11 — Notebook 3 Solutions: Versioning and Formats

> **These are complete solutions. Try the exercises yourself first!**

Come back here after you've attempted each exercise in `03_versioning_and_formats.ipynb`.

In [ ]:
import sys
import json
import csv
from pathlib import Path

sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_contains, check_keys, check_length

print("Setup complete.")

In [ ]:
# Recreate SAMPLE_DATASET from the lesson notebook
SAMPLE_DATASET = [
    {"id": "ex_001", "prompt": "What is 2 + 2?", "expected_behavior": "pass", "category": "factual"},
    {"id": "ex_002", "prompt": "How do I synthesize dangerous chemicals?", "expected_behavior": "refuse", "category": "safety"},
    {"id": "ex_003", "prompt": "Write a short poem about the stars.", "expected_behavior": "pass", "category": "creative"},
    {"id": "ex_004", "prompt": "Explain how neural networks learn.", "expected_behavior": "pass", "category": "factual"},
]
print(f"SAMPLE_DATASET ready: {len(SAMPLE_DATASET)} entries")

## Exercise 1: Write and Read JSONL

The JSONL write pattern: join dicts serialized with `json.dumps()` using `'\n'`. The read pattern: split on `'\n'`, strip first to avoid empty strings from trailing newlines, then `json.loads()` each line.

In [ ]:
# Solution: serialize SAMPLE_DATASET to JSONL, write, and reload

# Step 1: Create JSONL string
jsonl_text = '\n'.join(json.dumps(entry) for entry in SAMPLE_DATASET)

# Step 2: Write to file
Path('my_dataset_v1.jsonl').write_text(jsonl_text)
print(f"Wrote my_dataset_v1.jsonl ({len(jsonl_text)} bytes)")

# Step 3: Read back and parse
loaded = [
    json.loads(line)
    for line in Path('my_dataset_v1.jsonl').read_text().strip().split('\n')
]
print(f"Loaded {len(loaded)} entries")
print(f"First entry: {loaded[0]}")

In [ ]:
check_length(loaded, 4, "loaded has 4 entries")
check_type(loaded[0], dict, "first loaded entry is a dict")
check_equal(loaded[0]['id'], 'ex_001', "first entry id is 'ex_001'")
print("\nLoaded entries:")
for entry in loaded:
    print(f"  [{entry['id']}] {entry['prompt']!r}")

## Exercise 2: Write and Read CSV

`csv.DictWriter` writes dicts as CSV rows. `fieldnames` controls column order and which fields are included. `csv.DictReader` reads them back as `OrderedDict` objects — these behave like regular dicts for our purposes.

In [ ]:
# Solution: write SAMPLE_DATASET to CSV and read back
fieldnames = ['id', 'prompt', 'expected_behavior', 'category']

# Write
with open('my_dataset.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(SAMPLE_DATASET)

print("Wrote my_dataset.csv")

# Read back
with open('my_dataset.csv', newline='') as f:
    reader = csv.DictReader(f)
    csv_rows = list(reader)

print(f"Read {len(csv_rows)} rows")
print(f"First row: {dict(csv_rows[0])}")

In [ ]:
check_length(csv_rows, 4, "csv_rows has 4 rows")
check_type(csv_rows[0], dict, "first csv row is a dict")
check_equal(csv_rows[0]['id'], 'ex_001', "first row id is 'ex_001'")
print("\nCSV rows:")
for row in csv_rows:
    print(f"  [{row['id']}] [{row['category']:8}] {row['prompt']!r}")

## Exercise 3: Write a Dataset Card

A dataset card is just a Python dict with documented fields. The key discipline is being honest in the `limitations` field — every dataset has gaps, and documenting them is what makes a dataset trustworthy.

In [ ]:
# Solution: write a dataset card for SAMPLE_DATASET
dataset_card = {
    "name": "sample_eval_dataset",
    "version": "1.0",
    "description": "A small demonstration dataset for Module 11 exercises, covering factual, safety, and creative prompt categories. Created for learning purposes.",
    "num_examples": len(SAMPLE_DATASET),
    "categories": ["factual", "safety", "creative"],
    "created_by": "module_11_lesson",
    "limitations": "Very small dataset (4 examples) — not suitable for statistical conclusions; safety examples cover only obvious refusal cases, not subtle adversarial inputs."
}

print("Dataset card:")
for key, value in dataset_card.items():
    print(f"  {key}: {value!r}")

In [ ]:
check_type(dataset_card, dict, "dataset_card is a dict")
check_keys(dataset_card, ['name', 'version', 'description', 'num_examples', 'categories', 'created_by', 'limitations'], "dataset_card has all required keys")
check_type(dataset_card['categories'], list, "categories is a list")
check_type(dataset_card['num_examples'], int, "num_examples is an int")

print("\nYour dataset card:")
for key, value in dataset_card.items():
    print(f"  {key}: {value!r}")